---
# Phase 2 — Data Cleaning Pipeline

## Overview

The cleaning pipeline consists of ten sequential steps (N1–N10), each addressing a specific data quality issue identified during EDA. The steps are ordered to ensure that each transformation builds on a valid foundation:

```
N1: Remove NaN-labelled rows        → valid target variable
N2: Drop identifier columns         → remove non-predictive metadata
N3: Drop constant columns           → remove zero-variance features
N4: Encode Protocol (OHE)           → correct categorical treatment
N5: Impute missing values           → complete feature matrix
N6: Remove exact duplicates         → eliminate redundant observations
N7: Remove correlated features      → reduce dimensionality (domain-aware)
N8: Cap outliers (P01–P99)          → reduce extreme value influence
N9: Log-transform skewed features   → normalise distributions
N10: StandardScaler                  → equalise feature scales
```

**Design principle:** Each step is implemented with **explicit semantic awareness** — general-purpose heuristics are overridden by domain knowledge of network traffic wherever the two conflict (particularly for flag count features and directional asymmetry pairs).

---
## N1 — Initialisation & NaN Label Row Removal


In [58]:
print('='*70)
print('CLEANING N1 : INITIALISATION & NaN LABEL ROW REMOVAL')
print('='*70)

df_clean   = df.copy()
shape_init = df_clean.shape

# Remove rows where Label is NaN (unlabelled = unusable)
n_nan_label = df_clean['Label'].isna().sum()
df_clean    = df_clean.dropna(subset=['Label'])

print(f'\n  Initial shape              : {shape_init}')
print(f'  NaN Label rows removed     : {n_nan_label}')
print(f'  Shape after removal        : {df_clean.shape}')
print(f'  Unique labels              : {df_clean["Label"].unique()}')


CLEANING N1 : INITIALISATION & NaN LABEL ROW REMOVAL

  Initial shape              : (5222, 85)
  NaN Label rows removed     : 1
  Shape after removal        : (5221, 85)
  Unique labels              : ['Benign']


**Rationale:**
Rows with a missing `Label` cannot contribute to model evaluation, as the ground truth class is unknown. Removing them at the earliest stage prevents them from contaminating downstream statistics (e.g., distribution plots, correlation matrices) and ensures a clean working copy `df_clean` throughout the pipeline.


---
## N2 — Drop Network Identifier Columns & Encode Destination Port

**Columns removed:** Network metadata columns (Flow ID, Source/Destination IP, Timestamp, External IP, Source Port) are dropped because they serve as unique identifiers or carry no repeatable predictive signal:
- `Flow ID`: Arbitrary flow index with no statistical structure.
- `Source IP` / `Destination IP`: Anonymised or hashed in this dataset; even when available, raw IPs generalise poorly.
- `Timestamp`: Temporal ordering is not exploited by the models in this pipeline.
- `Source Port`: Ephemeral random port assigned by the OS; carries no predictive information.

**Destination Port — Special Treatment:** Unlike source ports, the destination port encodes the **targeted service** (HTTP=80, HTTPS=443, SSH=22, RDP=3389), which is strongly correlated with attack type. However, port numbers are **not ordinal** — treating them as continuous integers would impose a meaningless numeric relationship. The encoding strategy is:
- Top-N (N=10) most frequent destination ports → individual OHE binary columns.
- All remaining ports → aggregated into a single `Port_Other` column.
- The original integer column is dropped after encoding.

This approach balances expressiveness (preserving the most common service targets) with parsimony (avoiding one column per port number).


In [59]:
print('='*70)
print('CLEANING N2 : DROP NETWORK IDENTIFIER COLUMNS')
print('='*70)

# Technical identifiers — no predictive value
id_cols_config = {
    'Flow ID'        : 'flow identifier (index)',
    'Source IP'      : 'source IP address (anonymised/hashed)',
    'Destination IP' : 'destination IP address',
    'Timestamp'      : 'capture timestamp',
    'External IP'    : 'external IP (100% = 0.0 in this dataset)',
    'Source Port'    : 'ephemeral source port (random, not predictive)',
}

before = df_clean.shape
cols_to_drop_ids = [c for c in id_cols_config if c in df_clean.columns]

for col in cols_to_drop_ids:
    print(f'  ✗ {col:<20} → {id_cols_config[col]}')

df_clean.drop(columns=cols_to_drop_ids, inplace=True)
print(f'\n  Shape: {before} → {df_clean.shape}  ({before[1]-df_clean.shape[1]} columns dropped)')

# ── Destination Port: categorical treatment ───────────────────────────────────
# Port numbers are NOT ordinal/continuous. Port 443 is not "bigger" than port 80.
# They encode the targeted service → categorical signal for attack classification.
# Strategy: encode top-N most frequent ports as dummies, rest → 'Port_Other'
if 'Destination Port' in df_clean.columns:
    TOP_N_PORTS = 10
    top_ports = df_clean['Destination Port'].value_counts().nlargest(TOP_N_PORTS).index.tolist()

    def map_port(p):
        if p in top_ports:
            return f'Port_{int(p)}'
        return 'Port_Other'

    df_clean['Destination Port Cat'] = df_clean['Destination Port'].map(map_port)
    port_dummies = pd.get_dummies(df_clean['Destination Port Cat'],
                                  prefix='', prefix_sep='', dtype=int)
    df_clean = pd.concat([df_clean, port_dummies], axis=1)
    df_clean.drop(columns=['Destination Port', 'Destination Port Cat'], inplace=True)

    print(f'\n  Destination Port → OHE top-{TOP_N_PORTS} ports + Port_Other')
    print(f'  New columns: {list(port_dummies.columns)}')
    print(f'  Shape after port OHE: {df_clean.shape}')

# OHE port columns to exclude from capping / log transform / scaling
PORT_OHE_COLS = [c for c in df_clean.columns if c.startswith('Port_')]
print(f'\n  Port OHE columns (excluded from capping/log/scaling): {len(PORT_OHE_COLS)}')


CLEANING N2 : DROP NETWORK IDENTIFIER COLUMNS
  ✗ Flow ID              → flow identifier (index)
  ✗ Source IP            → source IP address (anonymised/hashed)
  ✗ Destination IP       → destination IP address
  ✗ Timestamp            → capture timestamp
  ✗ External IP          → external IP (100% = 0.0 in this dataset)
  ✗ Source Port          → ephemeral source port (random, not predictive)

  Shape: (5221, 85) → (5221, 79)  (6 columns dropped)

  Destination Port → OHE top-10 ports + Port_Other
  New columns: ['Port_0', 'Port_123', 'Port_1883', 'Port_1900', 'Port_443', 'Port_53', 'Port_54102', 'Port_80', 'Port_8080', 'Port_89', 'Port_Other']
  Shape after port OHE: (5221, 89)

  Port OHE columns (excluded from capping/log/scaling): 11


**Post-step verification:** The shape reduction confirms the number of identifier columns dropped. The new `Port_*` OHE columns are listed and will be explicitly **excluded** from outlier capping, log transformation, and StandardScaler in subsequent steps.


---
## N3 — Drop Constant & Zero-Variance Columns

**Rationale:** Features with zero variance (identical value across all observations) carry no discriminative information by definition — they cannot contribute to separating benign from anomalous flows. Their presence wastes memory and computation without any modelling benefit.

**Identification criterion:** `std = 0` computed on the cleaned dataset after NaN label removal. This recomputation is necessary because the EDA was conducted on the full raw dataset; NaN removal may alter variance estimates.

**Expected columns:** Based on the TCP flag analysis (Step 8 of EDA), flag columns with constant zero values (e.g., `URG Flag Count`, `CWE Flag Count`, `ECE Flag Count`) are the primary candidates for removal.


In [60]:
print('='*70)
print('CLEANING N3 : DROP CONSTANT COLUMNS')
print('='*70)

num_cols_clean = df_clean.select_dtypes(include=np.number).columns.tolist()
before = df_clean.shape

# Columns with std = 0 (constant)
zero_var = [c for c in num_cols_clean if df_clean[c].std() == 0]

print(f'\n  Constant columns (std=0): {len(zero_var)}')
for c in zero_var:
    print(f'    ✗ {c:<40} val={df_clean[c].unique()[0]}')

df_clean.drop(columns=zero_var, inplace=True)
print(f'\n  Shape: {before} → {df_clean.shape}  ({len(zero_var)} columns dropped)')

# Recompute numerical column list after dropping
num_cols_clean = df_clean.select_dtypes(include=np.number).columns.tolist()
# Exclude Label
feature_cols = [c for c in num_cols_clean if c != 'Label']


CLEANING N3 : DROP CONSTANT COLUMNS

  Constant columns (std=0): 9
    ✗ Bwd PSH Flags                            val=0
    ✗ Fwd URG Flags                            val=0
    ✗ Bwd URG Flags                            val=0
    ✗ URG Flag Count                           val=0.0
    ✗ CWE Flag Count                           val=0.0
    ✗ ECE Flag Count                           val=0.0
    ✗ Fwd Avg Bytes/Bulk                       val=0.0
    ✗ Fwd Avg Packets/Bulk                     val=0.0
    ✗ Fwd Avg Bulk Rate                        val=0.0

  Shape: (5221, 89) → (5221, 80)  (9 columns dropped)


**Post-step note:** After dropping constant columns, the numerical feature list `feature_cols` is recomputed. All subsequent steps operate on this updated feature set, ensuring no stale references to removed columns.


---
## N4 — Protocol One-Hot Encoding

**Rationale:** `Protocol` is stored as an integer (0, 6, 17) but represents a **nominal categorical variable**: TCP (6), UDP (17), and Unknown/Other (0) are distinct communication paradigms with no inherent ordering. Treating this feature as a continuous variable would incorrectly imply, for example, that UDP (17) is "2.8× more protocol" than TCP (6).

**Encoding choice — `drop_first=False`:** All three categories are explicitly retained as separate OHE columns (`Proto_TCP`, `Proto_UDP`, `Proto_Other`). The `Proto_Other` column (protocol = 0, unknown traffic) is preserved because:
- In 6G IoT environments, unrecognised protocol codes may indicate encapsulation, tunnelling, or evasion techniques.
- Silently dropping it with `drop_first=True` would discard this potential anomaly signal.

**OHE columns are excluded from all subsequent numerical transformations** (scaling, log transformation, outlier capping), as applying these operations to binary indicator variables is semantically meaningless.


In [61]:
print('='*70)
print('CLEANING N4 : PROTOCOL ENCODING (ONE-HOT)')
print('='*70)

if 'Protocol' in df_clean.columns:
    proto_map_str = {6: 'TCP', 17: 'UDP', 0: 'Other'}
    df_clean['Protocol_label'] = df_clean['Protocol'].map(
        lambda x: proto_map_str.get(x, f'Proto_{int(x)}')
    )

    before_shape = df_clean.shape
    # drop_first=False: keep all 3 categories (TCP / UDP / Other)
    # 'Other' (protocol=0) = unknown traffic → potential anomaly signal in IoT/6G
    # drop_first=True would silently drop one category depending on pandas sort order
    ohe_proto = pd.get_dummies(df_clean['Protocol_label'],
                               prefix='Proto', drop_first=False, dtype=int)
    df_clean  = pd.concat([df_clean, ohe_proto], axis=1)
    df_clean.drop(columns=['Protocol', 'Protocol_label'], inplace=True)

    feature_cols = [c for c in feature_cols if c != 'Protocol'] + list(ohe_proto.columns)

    print(f'\n  Protocol one-hot encoded (drop_first=False) → {list(ohe_proto.columns)}')
    print(f'  Shape: {before_shape} → {df_clean.shape}')
    print(f'  ℹ  Proto_Other kept: anomaly signal for unknown traffic (protocol=0)')


CLEANING N4 : PROTOCOL ENCODING (ONE-HOT)

  Protocol one-hot encoded (drop_first=False) → ['Proto_Other', 'Proto_TCP', 'Proto_UDP']
  Shape: (5221, 81) → (5221, 82)
  ℹ  Proto_Other kept: anomaly signal for unknown traffic (protocol=0)


**Post-step note:** The original `Protocol` integer column and the intermediate `Protocol_label` string column are both dropped. The OHE columns are added to `feature_cols` and flagged for exclusion from numerical preprocessing steps.


---
## N5 — Missing Value Imputation

**Context:** Two categories of missing values are addressed:

**A) Informative missingness — Rate-derived features (`Flow Bytes/s`, `Flow Packets/s`):**
These features are computed as `Total Bytes / Flow Duration`. When `Flow Duration ≈ 0` (typical in SYN flood or port scan flows that are terminated immediately), the result is `Inf` or `NaN`. This missingness pattern is *not random* — it is mechanistically linked to the attack behaviour itself. A binary `_was_missing` indicator is created **before** imputation to preserve this signal for the model.

**B) Standard missingness — All other numerical columns:**
Median imputation is applied (as opposed to mean imputation) because:
- The median is **robust to outliers** — a key property for network traffic features with heavy tails.
- It preserves the central tendency without being distorted by extreme values that are common in this dataset.

**No column deletion for missing values:** Given the low overall missing rate (< 1%), column deletion would be unnecessarily destructive.


In [62]:
print('='*70)
print('CLEANING N5 : NUMERICAL MISSING VALUE IMPUTATION')
print('='*70)

# Rate-derived features: NaN = degenerate flow (Duration ≈ 0 → Inf/NaN)
# Missingness is INFORMATIVE (port scan, SYN flood pattern) → create indicator first
RATE_DERIVED_COLS = ['Flow Bytes/s', 'Flow Packets/s']

print('\n  [A] Informative missingness indicators (derived features):')
for col in RATE_DERIVED_COLS:
    if col in df_clean.columns:
        # Replace Inf with NaN first
        df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)
        n_miss_nan = df_clean[col].isna().sum()
        if n_miss_nan > 0:
            df_clean[f'{col}_was_missing'] = df_clean[col].isna().astype(int)
            print(f'  📍 {col:<30} : {n_miss_nan} NaN/Inf → indicator created')
        else:
            print(f'  ✓  {col:<30} : no missing values')

print('\n  [B] Median imputation (all remaining numerical columns):')
current_num_cols = df_clean.select_dtypes(include=np.number).columns
imputed = []

for col in current_num_cols:
    n_miss = df_clean[col].isna().sum()
    if n_miss > 0:
        med = df_clean[col].median()
        df_clean[col].fillna(med, inplace=True)
        imputed.append({'Column': col, 'N': n_miss, 'Median': med})
        print(f'  ✓ {col:<40} {n_miss} NaN → median={med:.4f}')

total_missing_after = df_clean.isnull().sum().sum()
print(f'\n  Columns imputed       : {len(imputed)}')
print(f'  Missing values after  : {total_missing_after}')
print(f'  {"✓ No missing values remaining" if total_missing_after == 0 else "⚠ NaN values remain"}')


CLEANING N5 : NUMERICAL MISSING VALUE IMPUTATION

  [A] Informative missingness indicators (derived features):
  ✓  Flow Bytes/s                   : no missing values
  ✓  Flow Packets/s                 : no missing values

  [B] Median imputation (all remaining numerical columns):

  Columns imputed       : 0
  Missing values after  : 0
  ✓ No missing values remaining


**Post-step verification:** The zero total missing count after imputation confirms complete data integrity. The `_was_missing` indicator columns are registered for exclusion from scaling and log transformation in subsequent steps.


---
## N6 — Duplicate Detection & Removal

**Rationale:** Exact duplicate rows (identical values across all features and the label) can arise from:
- **CICFlowMeter logging artefacts:** The flow exporter may re-emit a flow record when a flow is re-keyed or a timeout is triggered.
- **Data aggregation errors:** Multiple CSV files concatenated during dataset construction.

**Impact on modelling:** Duplicate rows artificially inflate the training set and can bias the model towards over-representing certain flow patterns. They also inflate evaluation metrics if duplicates appear in both train and test sets.

**Criterion:** Exact row-level equality across *all columns simultaneously*, using `pandas.DataFrame.duplicated()`. Near-duplicate detection (based on approximate similarity) is beyond the scope of this pipeline.


In [63]:
print('='*70)
print('CLEANING N6 : DUPLICATE DETECTION & REMOVAL')
print('='*70)

n_before     = len(df_clean)
n_duplicates = df_clean.duplicated().sum()

print(f'\n  Exact duplicates found: {n_duplicates} ({n_duplicates/n_before*100:.2f}%)')

if n_duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f'  ✓ {n_duplicates} duplicates removed')
    print(f'  Shape: ({n_before}, {df_clean.shape[1]}) → {df_clean.shape}')
else:
    print('  ✓ No duplicates detected')

# Reset index
df_clean.reset_index(drop=True, inplace=True)
print(f'\n  Index reset. Final shape: {df_clean.shape}')


CLEANING N6 : DUPLICATE DETECTION & REMOVAL

  Exact duplicates found: 20 (0.38%)
  ✓ 20 duplicates removed
  Shape: (5221, 82) → (5201, 82)

  Index reset. Final shape: (5201, 82)


**Post-step note:** The index is reset after deduplication to ensure contiguous row indexing, which is required by downstream numpy operations (array slicing, mask indexing).


---
## N7 — Redundant Feature Removal via Correlation Analysis

**Objective:** Reduce feature dimensionality by removing features that are highly linearly redundant (|r| > 0.90) with another retained feature. This addresses the multicollinearity identified in EDA Step 5B.

**Problem with naive greedy removal:** A standard greedy algorithm (keep Feature A, drop Feature B based on column order) can inadvertently eliminate features that encode *directional network asymmetry* — the primary signal for many attack types.

**Domain-protected pairs:** The following feature pairs are explicitly protected from automatic removal, regardless of their correlation value:

| Protected Pair | Attack Signal |
|---|---|
| `Total Fwd Packets` ↔ `Total Backward Packets` | Bidirectional asymmetry → scans, reflection DDoS |
| `Fwd Packet Length Mean` ↔ `Bwd Packet Length Mean` | Payload size asymmetry → data exfiltration |
| `Flow Bytes/s` ↔ `Flow Packets/s` | Volume vs. rate → volume-based vs. rate-based attacks |
| `Fwd Header Length` ↔ `Bwd Header Length` | Header overhead per direction → encapsulation evasion |

**Corrected strategy:** Greedy removal is applied only to features *outside* the protected set. Both members of a protected pair are always retained.


In [64]:
print('='*70)
print('CLEANING N7 : DROP REDUNDANT FEATURES (r > 0.90)')
print('='*70)

feat_only = [c for c in df_clean.columns if c != 'Label']
corr_mat  = df_clean[feat_only].corr().abs()
upper     = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))

# ── Asymmetric features to PROTECT (never auto-drop either member of a pair) ──
# These pairs encode directionality — the asymmetry IS the IDS signal
PROTECTED_PAIRS = [
    ('Total Fwd Packets',      'Total Backward Packets'),   # bidirectional asymmetry
    ('Fwd Packet Length Mean', 'Bwd Packet Length Mean'),   # payload size per direction
    ('Flow Bytes/s',           'Flow Packets/s'),            # rate type differs: volume vs count
    ('Fwd Header Length',      'Bwd Header Length'),         # header overhead per direction
]
protected_cols = set()
for a, b in PROTECTED_PAIRS:
    if a in feat_only and b in feat_only:
        protected_cols.add(a)
        protected_cols.add(b)

print(f'\n  Protected columns (network asymmetry signal): {sorted(protected_cols)}')

# ── Greedy drop with domain protection ────────────────────────────────────────
to_drop = []
for col in upper.columns:
    if col in protected_cols:
        continue  # Never auto-drop a protected directional feature
    if any(upper[col] > 0.90):
        max_corr_col = upper[col].idxmax()
        max_corr_val = upper[col].max()
        if max_corr_col not in protected_cols or col not in protected_cols:
            to_drop.append((col, max_corr_col, max_corr_val))

before = df_clean.shape
print(f'\n  Redundant columns identified (outside protected features): {len(to_drop)}')
print(f'  {"Feature to drop":<40} {"Correlated with":<40} {"r":>6}')
print('  ' + '-'*90)
for col, partner, val in to_drop:
    print(f'  ✗ {col:<40} {partner:<40} {val:>6.4f}')

drop_names = [col for col, _, _ in to_drop]
df_clean.drop(columns=drop_names, inplace=True)
print(f'\n  Shape: {before} → {df_clean.shape}  ({len(drop_names)} columns dropped)')

feature_cols = [c for c in df_clean.columns if c != 'Label']


CLEANING N7 : DROP REDUNDANT FEATURES (r > 0.90)

  Protected columns (network asymmetry signal): ['Bwd Header Length', 'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Fwd Header Length', 'Fwd Packet Length Mean', 'Total Backward Packets', 'Total Fwd Packets']

  Redundant columns identified (outside protected features): 27
  Feature to drop                          Correlated with                               r
  ------------------------------------------------------------------------------------------
  ✗ Total Length of Fwd Packets              Total Fwd Packets                        0.9144
  ✗ Total Length of Bwd Packets              Total Backward Packets                   0.9606
  ✗ Bwd Packet Length Std                    Bwd Packet Length Max                    0.9422
  ✗ Fwd IAT Total                            Flow Duration                            0.9848
  ✗ Fwd IAT Max                              Flow IAT Max                             0.9658
  ✗ Bwd IAT 

**Post-step assessment:** The number of dropped columns confirms the degree of statistical redundancy present in CICFlowMeter features (which compute min/max/mean/std for multiple measurements). The shape reduction quantifies the dimensionality improvement while the protected pairs ensure detection-relevant asymmetry signals are preserved.


---
## N8 — Outlier Treatment via Percentile Capping (P01–P99)

**Strategy: Capping (Winsorisation), not removal.**

In network intrusion detection, removing outlier rows is methodologically inadvisable because:
- Extreme values may represent rare but legitimate traffic patterns (bulk file transfers, video streaming bursts).
- More critically, **attack flows are statistically extreme by design** — their outlier status is the detection signal, not noise.

**Percentile capping** (Winsorisation at the 1st and 99th percentile) clips extreme values to the fence boundaries, preserving row count and dampening the influence of the most extreme observations on scaling and distance calculations, without eliminating any flows.

**Explicit exclusions from capping:**

| Excluded Group | Reason |
|---|---|
| **Flag count features** (`*Flag Count`) | Extreme values ARE the attack signal (e.g., high SYN count = SYN flood). Capping would destroy exactly the pattern the model needs to detect. |
| **OHE columns** (`Proto_*`, `Port_*`) | Binary 0/1 variables; capping at P01–P99 has no semantic meaning on binary data. |


In [65]:
print('='*70)
print('CLEANING N8 : OUTLIER CAPPING (PERCENTILE 1%–99%)')
print('='*70)

# Flag counts: discrete integers whose extreme values ARE the attack signal
# (high SYN count = SYN flood, high RST count = scan/teardown pattern)
# Capping at P99 would destroy exactly the information needed for anomaly detection
FLAG_COUNT_COLS = [c for c in df_clean.columns if 'Flag Count' in c]

# OHE binary columns: capping makes no semantic sense on 0/1 variables
OHE_COLS = [c for c in df_clean.columns if c.startswith('Proto_') or c.startswith('Port_')]

# Columns excluded from capping
EXCLUDED_FROM_CAP = set(FLAG_COUNT_COLS + OHE_COLS)

current_num = [c for c in df_clean.select_dtypes(include=np.number).columns
               if c != 'Label'
               and df_clean[c].std() > 0
               and c not in EXCLUDED_FROM_CAP]

print(f'\n  Excluded from capping:')
print(f'    Flag counts ({len(FLAG_COUNT_COLS)}): {FLAG_COUNT_COLS}')
print(f'    OHE cols    ({len(OHE_COLS)}): {OHE_COLS[:5]}{"..." if len(OHE_COLS)>5 else ""}')

print(f'\n  {"Feature":<40} {"P01":>8} {"P99":>10} {"N_capped":>9} {"Mean before":>11} {"Mean after":>10}')
print('  ' + '-'*93)

for col in current_num:
    p01  = df_clean[col].quantile(0.01)
    p99  = df_clean[col].quantile(0.99)
    n_out = ((df_clean[col] < p01) | (df_clean[col] > p99)).sum()
    mean_before = df_clean[col].mean()
    df_clean[col] = df_clean[col].clip(p01, p99)
    mean_after  = df_clean[col].mean()
    if n_out > 0:
        print(f'  {col:<40} {p01:>8.2f} {p99:>10.2f} {n_out:>9} {mean_before:>11.2f} {mean_after:>10.2f}')

print('\n  ✓ Capping complete')
print(f'  ✓ {len(FLAG_COUNT_COLS)} flag count columns preserved intact (attack signal)')


CLEANING N8 : OUTLIER CAPPING (PERCENTILE 1%–99%)

  Excluded from capping:
    Flag counts (4): ['FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count']
    OHE cols    (12): ['Port_0', 'Port_123', 'Port_1883', 'Port_1900', 'Port_443']...

  Feature                                       P01        P99  N_capped Mean before Mean after
  ---------------------------------------------------------------------------------------------
  Flow Duration                              127.00 119042070.00       103 16186816.21 16183014.48
  Total Fwd Packets                            1.00    2762.00        52      106.17      50.13
  Total Backward Packets                       0.00    3371.00        52      128.02      59.46
  Fwd Packet Length Max                        0.00    1400.00        45      305.98     305.50
  Fwd Packet Length Min                        0.00     485.00        43       34.51      29.03
  Fwd Packet Length Mean                       0.00    1307.71      

**Interpretation of Capping Results:**
- The `N_capped` column counts the number of observations modified per feature.
- The `Mean before` vs. `Mean after` comparison quantifies the shift in central tendency caused by capping — a large shift indicates that extreme values were heavily influencing the distribution.
- Features with many capped observations (high N_capped) confirm the heavy-tailed nature identified in EDA.


---
## N9 — Log Transformation (Semantics-Aware)

**Objective:** Reduce right-skew in continuous flow features to approximate symmetry, thereby improving the stability of distance-based computations in the Autoencoder and Isolation Forest.

**Transformation strategy by feature group:**

| Feature Group | Transformation | Rationale |
|---|---|---|
| Continuous flow metrics (duration, rates, packet lengths, IAT) | `log1p` → Box-Cox fallback if |skew| > 1 remains | Multiplicative-scale measurements; log is the canonical transformation for lognormal-like distributions |
| **Flag counts** (`*Flag Count`) | **Excluded** | Discrete integer counters. `log1p(0) = 0`, `log1p(1) ≈ 0.693` — compresses the only two meaningful values and destroys the zero vs. non-zero distinction. High structural skewness is inherent to sparse discrete features and should not be corrected. |
| **OHE columns** (`Proto_*`, `Port_*`) | **Excluded** | Binary 0/1 — log transformation is not applicable. |
| **`_was_missing` indicators** | **Excluded** | Binary indicators — same reasoning as OHE. |

**Box-Cox fallback:** Applied to features that remain highly skewed (|skew| > 1) after `log1p`. Box-Cox finds the optimal power parameter λ that best normalises the distribution.

**Threshold:** |skewness| > 1.0 (highly skewed category from Step 4B).


In [66]:
print('='*70)
print('CLEANING N9 : LOG TRANSFORMATIONS (semantics-aware)')
print('='*70)

# ── Feature groups ─────────────────────────────────────────────────────────────

# Group A: continuous flow metrics → log1p + Box-Cox fallback if still skewed
# These are multiplicative-scale measurements where log is semantically valid

# Group B: flag counts → EXCLUDED
# Discrete 0/1/2... integers. High skew is structural (most flows = 0 flags).
# log1p collapses the only two meaningful values. Extreme values = attack signal.
FLAG_COUNT_COLS = [c for c in df_clean.columns if 'Flag Count' in c]

# Group C: OHE + indicator columns → EXCLUDED (binary / categorical)
OHE_COLS        = [c for c in df_clean.columns if c.startswith('Proto_') or c.startswith('Port_')]
INDICATOR_COLS  = [c for c in df_clean.columns if c.endswith('_was_missing')]

EXCLUDED_FROM_LOG = set(FLAG_COUNT_COLS + OHE_COLS + INDICATOR_COLS)

current_num = [c for c in df_clean.select_dtypes(include=np.number).columns
               if c != 'Label'
               and df_clean[c].std() > 0
               and c not in EXCLUDED_FROM_LOG]

SKEW_THRESHOLD = 1.0

print(f'\n  Excluded from log transformation:')
print(f'    Flag counts ({len(FLAG_COUNT_COLS)}): {FLAG_COUNT_COLS}')
print(f'    OHE cols    ({len(OHE_COLS)}): {OHE_COLS[:4]}{"..." if len(OHE_COLS)>4 else ""}')
print(f'    Indicators  ({len(INDICATOR_COLS)}): {INDICATOR_COLS}')
print()

log_applied = []
print(f'  {"Feature":<40} {"Skew before":>11} {"Skew log1p":>11} {"Skew final":>11} {"Method":<20}')
print('  ' + '-'*97)

for col in current_num:
    s_before = df_clean[col].skew()
    if abs(s_before) <= SKEW_THRESHOLD:
        continue

    log_data = np.log1p(df_clean[col].clip(lower=0))
    s_log    = log_data.skew()

    if abs(s_log) > SKEW_THRESHOLD:
        # Box-Cox fallback for features still skewed after log1p
        try:
            bc_data, _ = boxcox(log_data + 1e-6)
            s_final = pd.Series(bc_data).skew()
            df_clean[col] = bc_data
            method = 'log1p + BoxCox'
        except Exception:
            df_clean[col] = log_data
            s_final = s_log
            method = 'log1p (BC failed)'
    else:
        df_clean[col] = log_data
        s_final = s_log
        method = 'log1p'

    log_applied.append(col)
    print(f'  {col:<40} {s_before:>11.3f} {s_log:>11.3f} {s_final:>11.3f} {method:<20}')

print(f'\n  Features transformed          : {len(log_applied)}')
print(f'  Flag counts preserved intact  : {len(FLAG_COUNT_COLS)}')
print('  ✓ Transformations complete')


CLEANING N9 : LOG TRANSFORMATIONS (semantics-aware)

  Excluded from log transformation:
    Flag counts (4): ['FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count']
    OHE cols    (12): ['Port_0', 'Port_123', 'Port_1883', 'Port_1900']...
    Indicators  (0): []

  Feature                                  Skew before  Skew log1p  Skew final Method              
  -------------------------------------------------------------------------------------------------
  Flow Duration                                  1.922      -0.036      -0.036 log1p               
  Total Fwd Packets                              8.132       2.306       0.092 log1p + BoxCox      
  Total Backward Packets                         8.057       2.128      -0.812 log1p + BoxCox      
  Fwd Packet Length Max                          1.498      -0.385      -0.385 log1p               
  Fwd Packet Length Min                          5.329       0.205       0.205 log1p               
  Fwd Packet Lengt

**Interpretation:**
- The `Skew before` → `Skew log1p` → `Skew final` columns track the progressive reduction in skewness.
- Features where `log1p` reduces |skew| below 1.0 are considered sufficiently transformed (`Method = log1p`).
- Features requiring the Box-Cox fallback (`Method = Box-Cox`) had distributions too extreme or non-monotonic for log alone.
- The residual skewness after transformation will be further managed by StandardScaler in Step N10.


---
## N10 — Feature Scaling (StandardScaler)

**Objective:** Standardise the scale of continuous features to zero mean and unit variance. This is essential for:
- The **Autoencoder**: gradient descent convergence and activation function behaviour are highly sensitive to input scale.
- **Isolation Forest**: while trees are theoretically scale-invariant, the `contamination` parameter and anomaly score calibration are more stable on normalised data.

**Why StandardScaler (Z-score normalisation) over MinMaxScaler:**
After outlier capping and log transformation, the remaining distributions are approximately symmetric. StandardScaler is appropriate for this regime. MinMaxScaler would be sensitive to the remaining boundary values after capping.

**Explicit exclusions from scaling:**

| Excluded Group | Reason |
|---|---|
| **OHE columns** (`Proto_*`, `Port_*`) | Already on [0, 1] binary scale; scaling would distort their meaning |
| **`_was_missing` indicators** | Binary columns; scaling would introduce non-zero means for sparse signals |
| **Flag count columns** | Discrete integer counts with a semantically meaningful zero. Centering at the mean makes 0 (= no flag active) negative, destroying the primary discrimination between no-flag and flag-active flows |

**`df_final` assembly:** The scaled feature matrix `X_scaled` and the target `y` are explicitly combined into `df_final` before export to prevent `NameError` at the CSV export step.


In [67]:
print('='*70)
print('CLEANING N10 : FEATURE SCALING (STANDARDSCALER)')
print('='*70)

# Separate features and target
X = df_clean.drop(columns=['Label'])
y = df_clean['Label'].copy()

# Columns excluded from StandardScaler:
# 1. OHE binaries (Proto_, Port_) — already on [0,1] scale
# 2. _was_missing indicators — binary, scaling would distort interpretation
# 3. Flag counts — discrete counts with meaningful zero.
#    StandardScaler centers at mean → after scaling, 0 (= no flag) becomes negative.
#    This destroys the zero-vs-nonzero distinction that is the primary flag signal.
FLAG_COUNT_COLS = [c for c in X.columns if 'Flag Count' in c]
OHE_COLS        = [c for c in X.columns if c.startswith('Proto_') or c.startswith('Port_')]
INDICATOR_COLS  = [c for c in X.columns if c.endswith('_was_missing')]

cols_no_scale  = set(FLAG_COUNT_COLS + OHE_COLS + INDICATOR_COLS)
cols_to_scale  = [c for c in X.select_dtypes(include=np.number).columns
                  if c not in cols_no_scale]

scaler   = StandardScaler()
X_scaled = X.copy()
X_scaled[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

print(f'\n  Features scaled (StandardScaler)    : {len(cols_to_scale)}')
print(f'  OHE columns (not scaled)            : {len(OHE_COLS)}')
print(f'  Flag counts (not scaled)            : {len(FLAG_COUNT_COLS)} → {FLAG_COUNT_COLS}')
print(f'  _was_missing indicators (not scaled): {len(INDICATOR_COLS)}')
print(f'  Shape of X                          : {X_scaled.shape}')

print(f'\n  Verification after scaling (first 3 continuous columns):')
print(X_scaled[cols_to_scale[:3]].describe().loc[['mean','std','min','max']].round(4).to_string())

# ── Assemble df_final for export ──────────────────────────────────────────────
# df_final was never defined in the original notebook → NameError crash at export
df_final = X_scaled.copy()
df_final['Label'] = y.values

print(f'\n  ✓ Scaling complete')
print(f'  ✓ df_final assembled: {df_final.shape}  (features + Label)')


CLEANING N10 : FEATURE SCALING (STANDARDSCALER)

  Features scaled (StandardScaler)    : 38
  OHE columns (not scaled)            : 12
  Flag counts (not scaled)            : 4 → ['FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count']
  _was_missing indicators (not scaled): 0
  Shape of X                          : (5201, 54)

  Verification after scaling (first 3 continuous columns):
      Flow Duration  Total Fwd Packets  Total Backward Packets
mean         0.0000             0.0000                  0.0000
std          1.0001             1.0001                  1.0001
min         -2.5076            -1.3687                 -2.2239
max          1.6253             2.1802                  2.2984

  ✓ Scaling complete
  ✓ df_final assembled: (5201, 55)  (features + Label)


**Verification:**
The post-scaling statistics (mean ≈ 0, std ≈ 1) for the scaled continuous columns confirm correct application of StandardScaler. Excluded columns (OHE, flag counts, indicators) retain their original distributions.

**Pipeline Summary:**

| Step | Operation | Columns Affected |
|---|---|---|
| N1 | Drop NaN-labelled rows | Label column |
| N2 | Drop identifiers + OHE Destination Port | 6 identifier cols → top-10 port OHE |
| N3 | Drop constant columns | Zero-variance features |
| N4 | OHE Protocol | Protocol → Proto_TCP / Proto_UDP / Proto_Other |
| N5 | Impute missing values | Rate-derived cols (indicator) + all NaN cols (median) |
| N6 | Remove duplicates | All columns |
| N7 | Drop correlated features | Non-protected features with \|r\| > 0.90 |
| N8 | Cap outliers P01–P99 | Continuous features (excl. flags, OHE) |
| N9 | Log-transform skewed features | Continuous features with \|skew\| > 1 (excl. flags, OHE, indicators) |
| N10 | StandardScaler | Continuous scaled features (excl. flags, OHE, indicators) |


---
## Dataset Export

**Objective:** Persist the fully cleaned and preprocessed dataset to disk as `AIoT_6G_CLEANED.csv` for use in the modelling phase. This separation of EDA/cleaning from modelling ensures reproducibility — the modelling notebook can be re-executed independently from the cleaned file without rerunning the preprocessing pipeline.


In [68]:
print('='*70)
print('EXPORT CLEANED DATASET')
print('='*70)

output_file = 'AIoT_6G_CLEANED.csv'
df_final.to_csv(output_file, index=False)

import os
size_mb = os.path.getsize(output_file) / 1e6

print(f'\n  ✓ File exported: {output_file}')
print(f'  Shape          : {df_final.shape}')
print(f'  File size      : {size_mb:.2f} MB')

print('\n' + '='*70)
print('✓ EDA + CLEANING PIPELINE COMPLETED SUCCESSFULLY')
print('='*70)


EXPORT CLEANED DATASET

  ✓ File exported: AIoT_6G_CLEANED.csv
  Shape          : (5201, 55)
  File size      : 4.12 MB

✓ EDA + CLEANING PIPELINE COMPLETED SUCCESSFULLY


**Phase 1 Conclusion — EDA & Data Cleaning:**

The preprocessing pipeline successfully transforms the raw AIoT-Sol 6G dataset into a clean, model-ready feature matrix through 10 sequential, domain-aware steps. Key outcomes:
- **Dimensionality reduction:** Redundant, constant, and identifier columns removed; remaining features are semantically meaningful and non-redundant.
- **Distribution normalisation:** Log transformation reduces skewness; StandardScaler equalises feature scales for gradient-based and distance-based models.
- **Signal preservation:** Flag count features, directional asymmetry pairs, and missing value indicators are explicitly protected from transformations that would destroy their attack-detection relevance.
- **Data integrity:** All flows are retained (no row deletion except for unlabelled records and exact duplicates); outlier information is dampened via capping, not discarded.

The cleaned dataset is now ready for the anomaly detection modelling phase.
